src/features.py
# Preprocessing and Feature Engineering Pipeline

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

In [2]:
class FeatureEngineer:
    """Feature Engineering for Knowledge-Gap Mapping System"""
    
    def __init__(self):
        pass
    
    def preprocess(self, df: pd.DataFrame) -> pd.DataFrame:
        """Basic preprocessing and cleaning"""
        print("Starting preprocessing...")
        
        df = df.copy()
        
        # Convert timestamp (days since start) to proper format if needed
        if 'timestamp' in df.columns:
            df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
        
        # Remove invalid rows
        df = df.dropna(subset=['student_id'])
        df = df[df['num_interactions'] >= 0]
        
        print(f"Preprocessed data: {len(df):,} rows")
        return df
    
    def extract_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Extract behavioral, engagement, and performance-related features"""
        print("Extracting features per student and per concept...")
        
        # Group by student and concept (activity_type)
        grouped = df.groupby(['student_id', 'concept'])
        
        features = pd.DataFrame()
        
        # 1. Behavioral Features
        features['total_interactions'] = grouped['num_interactions'].sum()
        features['unique_days_active'] = grouped['timestamp'].nunique()
        features['avg_interactions_per_day'] = features['total_interactions'] / features['unique_days_active']
        
        # 2. Temporal / Persistence Features
        features['first_interaction_day'] = grouped['timestamp'].min()
        features['last_interaction_day'] = grouped['timestamp'].max()
        features['span_days'] = features['last_interaction_day'] - features['first_interaction_day']
        
        # 3. Engagement Intensity
        features['max_daily_clicks'] = grouped['num_interactions'].max()
        features['interaction_consistency'] = features['unique_days_active'] / (features['span_days'] + 1)  # avoid division by zero
        
        # Reset index to make student_id and concept columns
        features = features.reset_index()
        
        # 4. Global Student-level features (merge back)
        student_total = df.groupby('student_id').agg(
            total_student_interactions=('num_interactions', 'sum'),
            total_concepts_accessed=('concept', 'nunique')
        ).reset_index()
        
        features = features.merge(student_total, on='student_id', how='left')
        
        # 5. Normalized features (relative to student)
        features['interaction_ratio'] = features['total_interactions'] / features['total_student_interactions']
        
        print(f"✅ Extracted {features.shape[1]} features for {features['student_id'].nunique():,} students")
        return features
    
    def create_target(self, student_info: pd.DataFrame) -> pd.DataFrame:
        """Create target variable (proxy for knowledge gap) using final_result"""
        target = student_info[['id_student', 'final_result']].copy()
        target = target.rename(columns={'id_student': 'student_id'})
        
        # Binary target: 1 = knowledge gap / at-risk (Fail or Withdrawn), 0 = Pass/Distinction
        target['knowledge_gap'] = target['final_result'].apply(
            lambda x: 1 if x in ['Fail', 'Withdrawn'] else 0
        )
        
        print("Target distribution:")
        print(target['knowledge_gap'].value_counts())
        return target
    
    def build_full_feature_set(self, interaction_df: pd.DataFrame, student_info: pd.DataFrame = None) -> pd.DataFrame:
        """End-to-end feature engineering"""
        df_clean = self.preprocess(interaction_df)
        features = self.extract_features(df_clean)
        
        if student_info is not None:
            target = self.create_target(student_info)
            # Merge target to features (at student level)
            features = features.merge(target, on='student_id', how='left')
        
        print("\n🎉 Feature Engineering Complete!")
        print(f"Final feature shape: {features.shape}")
        print("Sample columns:", features.columns.tolist()[:10])
        
        return features

In [5]:
# ========================== QUICK TEST ==========================
if __name__ == "__main__":
    from src.ingestion import LogIngestion
    
    ingestor = LogIngestion()
    raw_df = ingestor.load_oulad_vle()
    enriched = ingestor.enrich_with_activity_type(raw_df)
    std_df = ingestor.standardize_logs(enriched)
    
    engineer = FeatureEngineer()
    feature_df = engineer.build_full_feature_set(std_df)
    print(feature_df.head())

Loading OULAD studentVle.csv (large file - using efficient loading)...
✅ Loaded 180,982 interaction records from AAA-2013J
✅ Enriched with activity_type from vle.csv
✅ Standardized DataFrame: 180,982 rows
Columns: ['student_id', 'timestamp', 'num_interactions', 'activity_type', 'concept']
Starting preprocessing...
Preprocessed data: 180,982 rows
Extracting features per student and per concept...
✅ Extracted 13 features for 378 students

🎉 Feature Engineering Complete!
Final feature shape: (2646, 13)
Sample columns: ['student_id', 'concept', 'total_interactions', 'unique_days_active', 'avg_interactions_per_day', 'first_interaction_day', 'last_interaction_day', 'span_days', 'max_daily_clicks', 'interaction_consistency']
   student_id    concept  total_interactions  unique_days_active  \
0       11391    forumng                 193                  19   
1       11391   homepage                 138                  40   
2       11391  oucontent                 553                  23   


In [7]:
# 03_feature_engineering_test.ipynb

from src.ingestion import LogIngestion
from src.features import FeatureEngineer
import pandas as pd

print("=== DAY 3: Preprocessing & Feature Engineering ===\n")

# 1. Load standardized data
ingestor = LogIngestion()
raw_df = ingestor.load_oulad_vle()
enriched = ingestor.enrich_with_activity_type(raw_df)
std_df = ingestor.standardize_logs(enriched)

print(f"Input shape: {std_df.shape}")

# 2. Feature Engineering
engineer = FeatureEngineer()
feature_df = engineer.build_full_feature_set(std_df)

# Optional: Load studentInfo for target (recommended)
try:
    student_info = pd.read_csv("data/studentInfo.csv")
    student_info = student_info[(student_info['code_module'] == 'AAA') & 
                               (student_info['code_presentation'] == '2013J')]
    feature_df = engineer.build_full_feature_set(std_df, student_info)
except:
    print("Could not load studentInfo.csv for target - continuing without target")

print("\nFinal Features Preview:")
print(feature_df.head())

print("\n🎉 DAY 3 SUCCESS! Feature engineering pipeline is working.")

=== DAY 3: Preprocessing & Feature Engineering ===

Loading OULAD studentVle.csv (large file - using efficient loading)...
✅ Loaded 180,982 interaction records from AAA-2013J
✅ Enriched with activity_type from vle.csv
✅ Standardized DataFrame: 180,982 rows
Columns: ['student_id', 'timestamp', 'num_interactions', 'activity_type', 'concept']
Input shape: (180982, 5)
Starting preprocessing...
Preprocessed data: 180,982 rows
Extracting features per student and per concept...
✅ Extracted 13 features for 378 students

🎉 Feature Engineering Complete!
Final feature shape: (2646, 13)
Sample columns: ['student_id', 'concept', 'total_interactions', 'unique_days_active', 'avg_interactions_per_day', 'first_interaction_day', 'last_interaction_day', 'span_days', 'max_daily_clicks', 'interaction_consistency']
Starting preprocessing...
Preprocessed data: 180,982 rows
Extracting features per student and per concept...
✅ Extracted 13 features for 378 students
Target distribution:
knowledge_gap
0    278
1